In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS schema.silver;

In [0]:
# load data
df_encounters = spark.read.table('schema.bronze.encounters')
df_patients = spark.read.table('schema.bronze.patients')
df_payers = spark.read.table('schema.bronze.payers')
df_procedures = spark.read.table('schema.bronze.procedures')
df_organizations = spark.read.table('schema.bronze.organizations')

In [0]:
from pyspark.sql.functions import col, to_date, regexp_replace, lower, trim

# Standardize encounters
print("Standardizing encounters table...")

# Column naming is already in snake_case
df_encounters_std = df_encounters

# Convert timestamp columns to date format (yyyy-MM-dd)
date_columns = ['start', 'stop']
for date_col in date_columns:
    if date_col in df_encounters_std.columns:
        df_encounters_std = df_encounters_std.withColumn(
            date_col,
            to_date(col(date_col))
        )

# Verify schema
print("\nStandardized schema:")
df_encounters_std.printSchema()
print("\nSample data:")
df_encounters_std.limit(3).display()

# Save to silver
df_encounters_std.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("schema.silver.encounters")
print("Saved Schema to Silver Layer")

In [0]:
from pyspark.sql.functions import to_date, coalesce

# Standardize patients
print("Standardizing patients table...")

# Column naming is already in snake_case
df_patients_std = df_patients

# Convert death_date from string to date format (yyyy-MM-dd)
# Try multiple date formats since the source format might vary
if 'death_date' in df_patients_std.columns:
    df_patients_std = df_patients_std.withColumn(
        'death_date',
        coalesce(
            to_date(col('death_date'), 'dd-MM-yyyy'),
            to_date(col('death_date'), 'yyyy-MM-dd')
        )
    )

# Verify schema
print("\nStandardized schema:")
df_patients_std.printSchema()
print("\nSample data:")
df_patients_std.limit(3).display()

# Save to silver
df_patients_std.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("schema.silver.patients")
print("Saved Schema to Silver Layer")

In [0]:
# Standardize payers and organizations
print("Standardizing payers table...")

# Column naming is already in snake_case
# No date/time columns to standardize
df_payers_std = df_payers
df_organizations_std = df_organizations

# Verify schema
print("\nStandardized schema for payers:")
df_payers_std.printSchema()
print("\nStandardized schema for organizations:")
df_organizations_std.printSchema()

print("\nSample data:")
df_payers_std.limit(3).display()

# Save to silver
df_payers_std.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("schema.silver.payers")
df_organizations_std.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("schema.silver.organizations")
print("Saved Schema to Silver Layer")

In [0]:
# Standardize procedures
print("Standardizing procedures table...")

# Column naming is already in snake_case
df_procedures_std = df_procedures

# Convert timestamp columns to date format (yyyy-MM-dd)
date_columns = ['start', 'stop']
for date_col in date_columns:
    if date_col in df_procedures_std.columns:
        df_procedures_std = df_procedures_std.withColumn(
            date_col,
            to_date(col(date_col))
        )

# Verify schema
print("\nStandardized schema:")
df_procedures_std.printSchema()
print("\nSample data:")
df_procedures_std.limit(3).display()

# Save to silver
df_procedures_std.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("schema.silver.procedures")
print("✓ Saved to schema.silver.procedures")

In [0]:
# Summary of standardization
print("="*70)
print("STANDARDIZATION COMPLETE")
print("="*70)

tables = ['encounters', 'patients', 'payers', 'procedures']

for table in tables:
    silver_table = f"schema.silver.{table}"
    df = spark.table(silver_table)
    row_count = df.count()
    print(f"\n{table.upper()}:")
    print(f"  ✓ Rows: {row_count:,}")
    print(f"  ✓ Columns: {len(df.columns)}")
    print(f"  ✓ Saved to: {silver_table}")

print("\n" + "="*70)
print("All tables standardized and saved to schema.silver")
print("="*70)